# MCP Practice

English | [Simplified Chinese](../../02_mcp/02_mcp-practice.ipynb)

This notebook is a guided walkthrough with Markdown code examples; the Chinese edition linked
above is the master text. Run the checked-in server scripts in a terminal. Course screenshots
are historical and may contain Chinese labels; current commands and configuration are given in
English below.

## MCP and Function Calling

Direct Function Calling often needs an application-specific wrapper for each external API. Repeating those wrappers across hosts creates integration work.

<img src="../../../../assets/FC06.png" alt="Original course illustration" style="width:50%;" />

Model Context Protocol standardizes discovery and exchange of external capabilities. Hosts can reuse compatible servers while retaining responsibility for model interaction, execution policy and user experience.

<img src="../../../../assets/FC07.png" alt="Original course illustration" style="width:50%;" />

## Call an MCP tool

### Basic demo

The examples use the [official Python SDK](https://github.com/modelcontextprotocol/python-sdk), pinned to `mcp>=1.12,<2`. The primary entry is `cases/00_mcp-server-demo/w-mcp_demo_server.py`. This minimal extract illustrates the structure:

```python
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("w-MCPDemo", host="127.0.0.1", port=16680)

@mcp.tool()
def add(a: int, b: int) -> int:
    """Add two integers."""
    return a + b

if __name__ == "__main__":
    mcp.run(transport="streamable-http")
```

See the [demo README](cases/00_mcp-server-demo/README.md) for the complete server and startup commands.

### OpenWeather MCP

Use `cases/01_weather-mcp-server/w-openweather_mcp_server.py`. The service exposes current weather and forecasts by city or coordinates, with English defaults and `language="zh"` for Chinese. Configure `OPENWEATHER_API_KEY` locally. Never include keys in committed client JSON.

See [setup and tool arguments](cases/01_weather-mcp-server/README.md). Call `get_weather` with a city such as `Nanjing`; call `get_forecast` with `forecast_date="tomorrow"` for the next local day. The forecast product samples every three hours, so its range is a sampled range rather than an official daily minimum/maximum.

## Three transport choices

| Transport | Deployment | Typical use |
| --- | --- | --- |
| stdio | Client-launched local process | Local desktop or CLI tools |
| Streamable HTTP | Independently running service | Shared or remote access |
| Legacy HTTP+SSE | Older HTTP integration | Compatibility with older clients |

### stdio

The client launches the server and exchanges JSON-RPC through stdin/stdout. Send diagnostics to stderr so they do not corrupt protocol messages. No HTTP listener is needed. Host support and configuration vary.

The checked-in `w-time_mcp_server.py` is the time example. A minimal standalone demonstration follows:

```python
from datetime import datetime, timezone
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("w-TimeMCP", host="127.0.0.1", port=16683)

@mcp.tool()
def get_current_time() -> str:
    """Return an aware UTC timestamp."""
    return datetime.now(timezone.utc).isoformat()

if __name__ == "__main__":
    mcp.run(transport="stdio")
```

Illustrative stdio host configuration. Replace executable and script paths with real absolute paths; configuration fields vary by host.

```json
{"mcpServers":{"w-TimeMCP":{"command":"python","args":["/absolute/path/to/w-time_mcp_server.py"]}}}
```

### Legacy HTTP+SSE

SSE carries a server-to-client event stream. The legacy MCP HTTP+SSE transport also uses HTTP submission for client messages, enabling protocol exchanges. Use it only when both endpoints support that legacy transport.

For transport comparison only, save this separately and run it instead of the HTTP example using the same port:

```python
from datetime import datetime, timezone
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("w-TimeMCP", host="127.0.0.1", port=16683)

@mcp.tool()
def get_current_time() -> str:
    """Return an aware UTC timestamp."""
    return datetime.now(timezone.utc).isoformat()

if __name__ == "__main__":
    mcp.run(transport="sse")
```

An illustrative legacy client entry uses a `w-TimeMCP` key and an SSE endpoint such as `http://127.0.0.1:16683/sse`. Check your host's legacy transport schema; a URL alone does not start the server.

### Streamable HTTP

Streamable HTTP carries MCP protocol messages over HTTP and can stream responses. It is not the same as language-model token streaming. The checked-in basic, weather and file services use this transport.

Minimal time example for HTTP transport comparison:

```python
from datetime import datetime, timezone
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("w-TimeMCP", host="127.0.0.1", port=16683)

@mcp.tool()
def get_current_time() -> str:
    """Return an aware UTC timestamp."""
    return datetime.now(timezone.utc).isoformat()

if __name__ == "__main__":
    mcp.run(transport="streamable-http")
```

Illustrative HTTP host configuration; use the schema supported by your client:

```json
{"mcpServers":{"w-TimeMCP":{"type":"streamableHttp","url":"http://127.0.0.1:16683/mcp"}}}
```

## File Workbench MCP

The server directory contains the primary Python script, sample text files, recovery state and operating guides. Keep persisted recovery data when changing startup paths.

<img src="../../../../assets/filemcp.png" alt="Original course illustration" style="width:20%;" />

Use `cases/02_file-mcp-server/w-file_workbench_mcp_server.py`. Its service identity is `w-FileWorkbenchMCP`, its default port is `16682`, and its transport is Streamable HTTP. Follow the [File Workbench README](cases/02_file-mcp-server/README.md) for the supported command and filesystem policy.

Start from the file server directory:

```bash
python -B w-file_workbench_mcp_server.py
```

After initialization, connect to `http://127.0.0.1:16682/mcp`. Console formatting and process identifiers depend on your environment.

<img src="../../../../assets/filemcp2.png" alt="Original course illustration" style="width:80%;" />

### Configure an MCP host

Cline is one possible host. Install it through your editor's extension mechanism, then open its MCP configuration UI. UI labels and supported transport fields vary by release; the historical screenshots below illustrate the workflow.

Add an HTTP connection to the running service. For a host supporting these fields, an illustrative entry is:

```json
{"mcpServers":{"w-FileWorkbenchMCP":{"type":"streamableHttp","url":"http://127.0.0.1:16682/mcp"}}}
```

Refresh the tool list after connecting. If your host uses another transport type spelling, follow its own schema.

<img src="../../../../assets/filemcp3.png" alt="Original course illustration" style="width:100%;" />

Verify that the service is connected and its tools are listed.

<img src="../../../../assets/filemcp4.png" alt="Original course illustration" style="width:40%;" />

### Configure a model

Choose an available model and provider in your host. Availability, pricing and tool support depend on your account and deployment.

<img src="../../../../assets/filemcp5.png" alt="Original course illustration" style="width:40%;" />

For a compatible local deployment or hosted provider, choose the host's OpenAI-compatible connection option and supply your own endpoint and credentials.

<img src="../../../../assets/filemcp6.png" alt="Original course illustration" style="width:40%;" />

### Exercise: inspect and rename files

Use a disposable working directory or create a snapshot before changing the sample files. Example request:

```text
List the sample files in the File Workbench directory, read their contents,
and suggest names and extensions that match each programming language.
Preview the rename plan before applying it.
```

Use the actual server-side path to `cases/02_file-mcp-server/files`. The service operates on its own machine's filesystem.

<img src="../../../../assets/filemcp7.png" alt="Original course illustration" style="width:100%;" />

<img src="../../../../assets/filemcp8.png" alt="Original course illustration" style="width:100%;" />

Example follow-up:

```text
Restore the directory to the snapshot taken before this exercise.
```

A snapshot must exist for a complete directory restore. Rename history alone cannot reconstruct every arbitrary edit. See the [showcase](cases/02_file-mcp-server/SHOWCASE.md) for the tested lifecycle and limitations.

<img src="../../../../assets/filemcp9.png" alt="Original course illustration" style="width:100%;" />

### Run a server in the background

On Linux/macOS, start from the server directory:

```bash
nohup python -B w-file_workbench_mcp_server.py > mcp_server.log 2>&1 &
```

Track the launched process and stop that specific process when finished. On Windows,
use `Start-Process` with the server directory as `-WorkingDirectory` and
`-WindowStyle Hidden`, or use your normal service manager. Redirect logs as needed.
Keep secrets out of command lines and do not stop unrelated Python processes.

## Runnable examples and checks

### Weather MCP

[Current weather, forecasts and setup](cases/01_weather-mcp-server/README.md)

### File MCP

[File operations and recovery](cases/02_file-mcp-server/README.md)

Run the offline suites listed in the [topic README](../README.md). They cover English and Chinese behavior without using real OpenWeather quota.

## Local verification record (2026-10-18, `env_agent_uv`)

The offline suites ran from their documented working directories: 26 weather and bilingual checks,
21 file-workbench checks and 4 Skill comparisons, 51 in total, all passing. They mock OpenWeather and
use temporary files, so no real API quota is consumed.

**Expected output:**

```text
# 02_mcp/cases
Ran 26 tests in 5.027s
OK
Ran 21 tests in 2.769s
OK

# 03_skills/cases/01_weather_skill
Ran 4 tests in 1.491s
OK
```

All four MCP services were started locally and called for real; the tool lists and values below are
actual results. `w-OpenWeatherMCP` reports `missing_api_key` when no key is configured, matching the
diagnostics from the direct Skill script.

**Expected output:**

```text
w-MCPDemo          streamable-http  tools: ping, add                       prompts: greet_user
w-TimeMCP          stdio            tools: get_current_time, convert_time
w-OpenWeatherMCP   streamable-http  tools: ping, check_config, get_weather,
                                      get_weather_by_coordinates, get_forecast,
                                      get_forecast_by_coordinates
w-FileWorkbenchMCP streamable-http  tools: 18 (ping, list_allowed_roots, list_files, read_file,
                                      rename_file, delete/restore and directory snapshots)

$ add(a=29, b=13)
42
$ read_resource("greeting://张三/zh")
你好，%E5%BC%A0%E4%B8%89！
$ get_current_time(timezone="Asia/Shanghai", language="zh")
{"ok": true, "current_time": "2026-10-18 22:36:35", "timezone": "CST", "utc_offset_seconds": 28800}
$ convert_time(iso_time="2026-10-18T18:30:00", from_timezone="Asia/Shanghai", to_timezone="America/New_York")
{"ok": true, "target_iso8601": "2026-10-18T06:30:00-04:00"}
$ list_files(base_dir="02_file-mcp-server/files", language="zh", limit=3)
{"ok": true, "count": 3, "total": 5}
$ check_config(language="zh")                      # with OPENWEATHER_API_KEY unset
{"ok": false, "error": "missing_api_key"}
```
